# Huấn luyện YOLOv8 phát hiện biển số xe Việt Nam — Nhóm 08

Train **YOLOv8s tìm vị trí biển số** trên bộ dữ liệu biển số Việt Nam
[tanhphp/vietnamese-license-plates](https://www.kaggle.com/datasets/tanhphp/vietnamese-license-plates)
(728 ảnh gốc, 964 biển, ô tô / xe máy / xe tải, 1 lớp `Bien-so`;
nguồn: [franceto/Dataset-Vietnamese-License-Plate](https://github.com/franceto/Dataset-Vietnamese-License-Plate)).

Kết quả: `weights/best.pt`. Đặt vào thư mục `weights/` trên máy (đọc ký tự do PaddleOCR đảm nhận, không cần huấn luyện ở đây).

Chạy trên **Google Colab**: menu **Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU**. Chạy lần lượt từng ô (Shift + Enter).

## 1. Gắn Google Drive, lấy code, cài thư viện

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/lpr_project'   # thư mục project trên Drive
ZIP = '/content/drive/MyDrive/lpr_project.zip'   # file zip đã upload lên Drive (nếu chưa giải nén)
if not os.path.exists(PROJECT) and os.path.exists(ZIP):
    !unzip -q -o "$ZIP" -d /content/drive/MyDrive/
assert os.path.exists(PROJECT), 'Không thấy project: upload lpr_project.zip vào thư mục gốc My Drive'
%cd $PROJECT
!pip install -q ultralytics albumentations kagglehub roboflow
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ - vào Runtime > Change runtime type > T4 GPU')

## 2. Tải bộ dữ liệu từ Kaggle
Dataset công khai nên tải được không cần tài khoản. Nếu báo lỗi đăng nhập: vào kaggle.com → Settings → **Create New Token**,
tải `kaggle.json` rồi bỏ dấu `#` ở 2 dòng dưới để upload.

In [ ]:
# from google.colab import files; files.upload()          # chỉ khi bị đòi đăng nhập: chọn kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
import kagglehub
DATA_RAW = kagglehub.dataset_download('tanhphp/vietnamese-license-plates')
print('Đã tải về:', DATA_RAW)
!find "$DATA_RAW" -maxdepth 3 -type d | head -20
!find "$DATA_RAW" -name data.yaml -exec cat {} \;

## 3. Gộp dữ liệu và chia train / val / test = 70 / 20 / 10
Script tự loại ảnh trùng và giữ các bản augment của cùng 1 ảnh gốc trong cùng 1 tập (tránh rò rỉ sang tập test).
Dòng *Class id gốc* phải chỉ có 1 lớp (`{0: ...}`).

In [ ]:
!python scripts/prepare_dataset.py --src "$DATA_RAW" --out /content/data/lp

In [ ]:
from IPython.display import Image, display
# Thống kê dữ liệu (số ảnh mỗi tập, tỉ lệ biển 1 dòng / 2 dòng, kích thước, vị trí) -> hình cho báo cáo
!python scripts/visualize.py dataset --data-dir /content/data/lp
display(Image('runs/figures/dataset_stats.png', width=1000))

## 4. Huấn luyện
- `yolov8s` chính xác hơn `yolov8n`, vẫn đủ nhanh cho app. Muốn webcam mượt hơn trên CPU thì đổi thành `yolov8n.pt`.
- 100 epoch trên T4 khoảng 20–40 phút (dữ liệu nhỏ). `patience=20`: tự dừng sớm nếu 20 epoch liền không cải thiện.
- Xong sẽ tự copy mô hình tốt nhất về `weights/best.pt` (trên Drive).

In [ ]:
!python scripts/train.py --data /content/data/lp/data.yaml --model yolov8s.pt --epochs 100 --batch 32 --device 0

In [ ]:
import os
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'val_batch0_pred.jpg']:
    p = f'runs/lp_yolov8/{f}'
    if os.path.exists(p): display(Image(p, width=900))

## 5. Đánh giá trên tập test (Precision, Recall, mAP@0.5, mAP@0.5:0.95)
Ghi lại các số này cho báo cáo.

In [ ]:
!python scripts/eval_detector.py --weights weights/best.pt --data /content/data/lp/data.yaml --device 0

## 6. Tải mô hình về máy
- `best.pt` → chép vào thư mục `weights/` của project trên máy.
- `lp_test.zip` (tập test) → dùng để gán nhãn OCR và đánh giá end-to-end (`scripts/eval_e2e.py`).

In [ ]:
!cd /content/data && zip -qr /content/lp_test.zip lp/images/test lp/labels/test lp/data.yaml
from google.colab import files
files.download('weights/best.pt')
files.download('/content/lp_test.zip')